# Loan Payback Prediction Modelling

## Package Version Check and Loading

In [ ]:
# !python --version

In [ ]:
!pip install skops
# !pip list

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier
# import itertools

import skops.io as sio

## package versions:
# scikit-learn 1.6.1
# numpy 2.0.2
# pandas 2.2.2
# skops 0.13.0

## Data Prep

In [ ]:
data = pd.read_csv('/kaggle/input/playground-series-s5e11/train.csv')
display(data.head(5))
print(len(data))

In [ ]:
## drop id from columns
data = data.drop('id', axis=1)

## remove duplicated data
data = data.drop_duplicates()
display(data.head(5))
print(len(data))

## inspect data types
display(data.info())

In [ ]:
## classify numerics and categorical
numerik = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount',
           'interest_rate']

objek = ['gender', 'marital_status', 'education_level', 'employment_status',
         'loan_purpose', 'grade_subgrade']

target = 'loan_paid_back'

In [ ]:
## remove data with target outside 0 or 1
data = data[(data[target]>=0) | (data[target]<=1)]
print(len(data))

In [ ]:
## remove negatives on numericals
for i in numerik:
    data = data[(data[i]>=0)]

print(len(data))

In [ ]:
## separate feature and target
X = data.copy().drop(target, axis=1)
display(X.head())

y = data[target].copy()
display(y.head())

## Quick EDA

In [ ]:
## look for nans
nan_df = X.isna().sum(0)
display(nan_df)

In [ ]:
## uniques
for i in objek:
    isi = X[i].unique()
    print(f"{i}: {isi}, len: {len(isi)}")

Notes:

* Other on gender, education level, and loan_purpose will be set as nan
* Education Level and grade is set as ordinals

In [ ]:
## uniques on numericals
for i in numerik:
    isi = X[i].unique()
    print(f"{i}. len: {len(isi)}")

Notes: No low unique count of numericals, so proceed as is (treat as numericals)

In [ ]:
## weights for balancing
balancing = (len(y) - sum(y))/sum(y)
print(balancing)

## Data preprocessing

In [ ]:
## perform ordinal encoding
g_class  = ['Male', 'Female']
ms_class = ['Single', 'Married', 'Divorced', 'Widowed']
el_class = ['High School', "Bachelor's", "Master's", 'PhD']
es_class = ['Self-employed', 'Employed', 'Unemployed', 'Retired', 'Student']
lp_class = ['Debt consolidation', 'Home', 'Education', 'Vacation', 'Car',
            'Medical', 'Business']
gs_class = X['grade_subgrade'].unique()
gs_class.sort()

X_ord = X[objek]
enc_cat = [g_class, ms_class, el_class, es_class, lp_class, gs_class]

enc = OrdinalEncoder(categories=enc_cat,
                     handle_unknown='use_encoded_value',
                     unknown_value=np.nan)
enc.fit(X_ord)
sio.dump(enc, "encoder.skops")

In [ ]:
## transform ordinal

## transform to ordinal encoded data
X_enc = enc.transform(X_ord)
X_enc = pd.DataFrame(X_enc, index=X.index,
                     columns=enc.get_feature_names_out())

## replace grade feature with encoded value
X = X.drop(objek, axis=1)
X = X.merge(X_enc, how='inner',
            left_index=True, right_index=True) ## use index as keys

display(X.head())

In [ ]:
cat_feat = [False, False, False, False, False,
            True, True, True, True, True, True]

## Modelling

In [ ]:
## prepare strateifiedKFold split
skf = StratifiedKFold()
skf.split(X, y)

def flag_taking(core_list, subset, flag_list):
    # 1. Create a lookup map: {item: index}
    # This makes finding the position of an item O(1) speed
    index_map = {item: idx for idx, item in enumerate(core_list)}
    
    # 2. Get the flag for each item in the subset
    # We find the index from the map, then look it up in the flag_list
    result_flags = [flag_list[index_map[item]] for item in subset if item in index_map]

    return result_flags

In [ ]:
## fill with columns for fixed features to consider
## or empty it so you can run seq. feature selection
best_col = []
# best_col  = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount',
#              'employment_status']


## defaults for feature selection
def_nest  = 1000
def_leaf  = 31
def_eta   = 0.1

## hyperparam setup
run_hyperparam = True
hyperparam_style = 'grid'
all_nest = [100, 120, 150, 200, 260, 350, 500, 600, 750, 1000]
all_leaf = [7, 15, 31, 63, 127, 255]
all_eta  = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

In [ ]:
## feature selection
if len(best_col) == 0:
    
    Xc = X.copy() ## make dummy
    cat_feat_c = cat_feat.copy()
    
    n_col = len(Xc.columns)
    n_split = 5
    
    ## Sequential FS
    all_col = list(Xc.columns)
    used_flag = [False]*n_col
    # print(used_flag)
    
    count = 0
    best_metric = 0
    best_flag = used_flag.copy()
    
    while count<2:
        new_best = False
        iter_metric = 0
        iter_flag = used_flag.copy()
        for i in range(n_col):
            if used_flag[i] == False:
                dummy = used_flag.copy()
                dummy[i] = True
                Xcc = Xc.iloc[:,dummy]
                cat_feat_cc = [item for item, flag in zip(cat_feat_c, dummy) if flag]
                # print(dummy)
                # display(Xcc.head())
                model = HistGradientBoostingClassifier(loss='log_loss',
                                                       learning_rate=def_eta,
                                                       max_iter=def_nest,
                                                       max_leaf_nodes=def_leaf,
                                                       categorical_features=cat_feat_cc,
                                                       random_state=42,
                                                       class_weight='balanced')
                all_cv = cross_val_score(model, Xcc, y,
                                         scoring='roc_auc',
                                         cv=skf)
                metric = np.mean(all_cv)
                if metric > iter_metric:
                    iter_metric = metric
                    iter_flag   = dummy.copy()
                if metric > best_metric:
                    print(metric)
                    best_metric = metric
                    best_flag   = dummy.copy()
                    new_best = True
        used_flag = iter_flag.copy()
        if new_best:
            count=0
        else:
            count += 1
        print(f"counter:{count}")
    
    best_col = [all_col[i] for i in range(len(best_flag)) if best_flag[i]]
    print(best_col)

best_flag = flag_taking(list(X.columns), best_col, cat_feat)
X = X[best_col].copy()
display(X.head())
print(list(X.columns))
print(best_flag)

In [ ]:
## param tuning
best_metric = 0
best_nest   = def_nest
best_leaf   = def_leaf
best_eta    = def_eta

# all_n_est = [1000, 1500, 2000]#, 3500, 5000, 7500, 10000]
if run_hyperparam:

    model = HistGradientBoostingClassifier(loss='log_loss',
                                           # learning_rate=i[2],
                                           max_iter=def_nest,
                                           # max_leaf_nodes=i[1],
                                           categorical_features=best_flag,
                                           random_state=42,
                                           class_weight='balanced')

    params = {'learning_rate': all_eta,
              'max_leaf_nodes': all_leaf}

    if hyperparam_style == 'grid':
        ht     = GridSearchCV(model, params,
                              scoring='roc_auc',
                              n_jobs=-1,
                              refit=False,
                              cv=skf
                              )        
    else:
        ht     = RandomizedSearchCV(model, params,
                                    n_iter=10,
                                    random_state=300,
                                    scoring='roc_auc',
                                    n_jobs=-1,
                                    refit=False,
                                    cv=skf
                                    )

    ht_res = ht.fit(X,y)

    # print(ht_res.best_params_)
    best_leaf = ht_res.best_params_['max_leaf_nodes']
    best_eta  = ht_res.best_params_['learning_rate']

    print('done')

In [ ]:
## final fit
print(f"best_nest : {best_nest}")
print(f"best_depth: {best_leaf}")
print(f"best_eta  : {best_eta}")

model = HistGradientBoostingClassifier(loss='log_loss',
                                       learning_rate=best_eta,
                                       max_iter=best_nest,
                                       max_leaf_nodes=best_leaf,
                                       categorical_features=best_flag,
                                       random_state=42,
                                       class_weight='balanced')

model.fit(X, y)
prediksi = model.predict_proba(X)
sio.dump(model, "model.skops")
print(prediksi[:,1])
skor_roc = roc_auc_score(y, prediksi[:,1])
print(skor_roc)

## Predict

In [ ]:
Xt = pd.read_csv('/kaggle/input/playground-series-s5e11/test.csv')

Xt_id = Xt['id'].copy()
Xt = Xt.drop('id',axis=1)
Xt = Xt.drop_duplicates()

## switch to ordinals
Xt_ord = Xt[objek]
## transform to ordinal encoded data
Xt_enc = enc.transform(Xt_ord)
Xt_enc = pd.DataFrame(Xt_enc, index=Xt.index,
                      columns=enc.get_feature_names_out())

## replace grade feature with encoded value
Xt = Xt.drop(objek, axis=1)
Xt = Xt.merge(Xt_enc, how='inner',
              left_index=True, right_index=True) ## use index as keys

display(Xt.head())

In [ ]:
## predict time
Xt = Xt[best_col].copy() ## feature selection

display(Xt.head())

prediksi = model.predict_proba(Xt)

pred_dict = {}
pred_dict['id'] = Xt_id
pred_dict[target] = prediksi[:,1]

pred_pd = pd.DataFrame(pred_dict)
display(pred_pd.head())

pred_pd.to_csv('pred.csv', index=False)